# Director-AI — Medical RAG Chatbot

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/06_medical_rag_chatbot.ipynb)

End-to-end walkthrough: load a medical knowledge base, ingest into vector store,
guard an LLM client, and ask medical questions with NLI-backed hallucination detection.

Covers:
1. Medical profile setup
2. Knowledge base ingestion
3. Guarded LLM queries
4. Evidence chunks and rejection handling
5. NLI scoring details

In [ ]:
!pip install -q director-ai

## 1. Configure the Medical Profile

The `medical` profile uses hybrid scoring (NLI + LLM judge), a 0.75 threshold,
and equal logic/fact weights.

In [ ]:
from director_ai.core.config import DirectorConfig

cfg = DirectorConfig.from_profile("medical")
print(f"Profile:    {cfg.profile}")
print(f"Threshold:  {cfg.coherence_threshold}")
print(f"Hard limit: {cfg.hard_limit}")
print(f"Backend:    {cfg.scorer_backend}")
print(f"NLI:        {cfg.use_nli}")
print(f"Reranker:   {cfg.reranker_enabled}")

## 2. Build Knowledge Base

Ingest medical facts into a VectorGroundTruthStore. In production, load from
clinical guidelines, drug databases, or PubMed abstracts.

In [ ]:
from director_ai.core import GroundTruthStore

store = GroundTruthStore()

medical_facts = {
    "aspirin": "Aspirin (acetylsalicylic acid) inhibits COX-1 and COX-2. Standard dose: 75-325mg daily for cardiovascular prophylaxis. Contraindicated in children under 16 (Reye syndrome risk).",
    "hypertension": "Hypertension is defined as systolic BP >= 140 mmHg or diastolic BP >= 90 mmHg (JNC 8). First-line treatment: ACE inhibitors, ARBs, calcium channel blockers, or thiazide diuretics.",
    "diabetes_a1c": "HbA1c target for most adults with type 2 diabetes: < 7.0% (ADA 2024). Metformin remains first-line pharmacotherapy. GLP-1 receptor agonists recommended for patients with ASCVD.",
    "penicillin": "Penicillin allergy reported in ~10% of patients; true IgE-mediated allergy confirmed in < 1% after skin testing. Cross-reactivity with cephalosporins is ~2%.",
    "stroke": "Acute ischemic stroke: IV alteplase within 4.5 hours of symptom onset (AHA/ASA 2019). Mechanical thrombectomy for large vessel occlusion up to 24 hours with favorable imaging.",
}

for key, fact in medical_facts.items():
    store.add(key, fact)

print(f"Ingested {len(medical_facts)} medical facts")

## 3. Build Scorer with Medical Profile

In [ ]:
from director_ai.core import CoherenceScorer

scorer = CoherenceScorer(
    threshold=cfg.coherence_threshold,
    hard_limit=cfg.hard_limit,
    soft_limit=cfg.soft_limit,
    use_nli=False,  # set True if torch+transformers installed
    ground_truth_store=store,
    w_logic=cfg.w_logic,
    w_fact=cfg.w_fact,
)
print("Scorer ready")

## 4. Score Medical Q&A Pairs

Test with accurate and hallucinated responses.

In [ ]:
test_pairs = [
    (
        "What is the first-line treatment for hypertension?",
        "First-line treatment includes ACE inhibitors, ARBs, calcium channel blockers, or thiazide diuretics.",
    ),
    (
        "What is the first-line treatment for hypertension?",
        "The recommended first-line treatment is homeopathic salt therapy and crystal healing.",
    ),
    (
        "Can children take aspirin?",
        "Aspirin is contraindicated in children under 16 due to Reye syndrome risk.",
    ),
    (
        "Can children take aspirin?",
        "Children of any age can safely take aspirin at adult doses.",
    ),
    (
        "What is the HbA1c target for type 2 diabetes?",
        "The ADA recommends an HbA1c target below 7.0% for most adults with type 2 diabetes.",
    ),
]

for prompt, response in test_pairs:
    approved, score = scorer.review(prompt, response)
    status = "PASS" if approved else "BLOCKED"
    print(f"[{status}] {response[:80]}...")
    print(
        f"  coherence={score.score:.3f}  h_log={score.h_logical:.2f}  h_fact={score.h_factual:.2f}",
    )
    if score.evidence:
        print(f"  evidence: {score.evidence[0].chunk[:60]}...")
    print()

## 5. Full Agent Pipeline

The `CoherenceAgent` orchestrates generation + scoring + halt logic.

In [ ]:
from director_ai.core import CoherenceAgent

agent = CoherenceAgent(_scorer=scorer, _store=store)
result = agent.process("What is the stroke treatment window?")

print(f"Output:  {result.output}")
print(f"Halted:  {result.halted}")
if result.coherence:
    print(f"Score:   {result.coherence.score:.3f}")

## Summary

- The `medical` profile applies stricter thresholds (0.75) with hybrid scoring
- Knowledge base ingestion grounds the scorer in real clinical data
- Hallucinated medical claims are reliably blocked
- Evidence chunks provide traceable rejection rationale

For production, install `director-ai[nli]` for DeBERTa-based NLI scoring.